# Detecting mink from audio — OOTB Perch / Bird classifier test

**216labs research** · See `paper_draft.md` and `citations.bib` in this repo folder.

**Runtime:** Colab **GPU** (Perch 2.0 / perch-hoplite).  
**Credentials:** Kaggle API for model weights (`KAGGLE_USERNAME`, `KAGGLE_KEY`).

## Protocol
1. Install deps
2. Run **bird vocalization classifier** (same as Bird Perch production)
3. Run **Perch 2.0** (`perch_v2`)
4. Upload mink (+ control) audio clips
5. Scan top-*k* labels for mink / mustelid / *Neogale* tokens
6. Record verdict in final cell

## 1. Installation

In [ ]:
#@title Install dependencies { display-mode: "form" }
!apt-get -qq install -y ffmpeg libsndfile1 > /dev/null
%pip install -q kagglehub tensorflow-cpu librosa soundfile scipy numpy
# Perch 2.0 (GPU + TF 2.20 rc per upstream README)
%pip install -q "tensorflow[and-cuda]~=2.20.0rc0"
%pip install -q git+https://github.com/google-research/perch-hoplite.git

In [ ]:
#@title Kaggle credentials (for bird classifier download) { display-mode: "form" }
import os
from google.colab import userdata

for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
if not os.environ.get("KAGGLE_KEY"):
    print("Set KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets, or export in environment.")

## 2. Model A — Bird vocalization classifier (Bird Perch production)

Kaggle: `google/bird-vocalization-classifier` TF2 v4. Avian label space.

In [ ]:
#@title Load bird classifier SavedModel { display-mode: "form" }
import glob
import re
import numpy as np
import tensorflow as tf
import kagglehub
import librosa

BIRD_HANDLE = "google/bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier/4"
bird_dir = kagglehub.model_download(BIRD_HANDLE)
print("Model dir:", bird_dir)

bird_model = tf.saved_model.load(bird_dir)
bird_infer = bird_model.signatures.get("serving_default") or next(iter(bird_model.signatures.values()))
spec_in = bird_infer.structured_input_signature[1] or {}

# Pick waveform input (largest fixed length, prefer name hints)
best = None
for name, tensor in spec_in.items():
    sh = tensor.shape.as_list()
    if len(sh) == 2 and sh[0] == 1 and sh[1] is not None:
        t = int(sh[1])
        score = float(t)
        if any(h in name.lower() for h in ("waveform", "audio", "input")):
            score += 1e9
        if best is None or score > best[0]:
            best = (score, t, name)
BIRD_WAVE_KEY = best[2] if best else next(iter(spec_in.keys()))
BIRD_SAMPLES = best[1] if best else 160000
print(f"Input key={BIRD_WAVE_KEY!r} samples={BIRD_SAMPLES}")

label_path = None
for pat in ("**/labels.txt", "**/*label*.txt", "**/assets/labels.csv"):
    hits = glob.glob(f"{bird_dir}/{pat}", recursive=True)
    if hits:
        label_path = hits[0]
        break
bird_labels = []
if label_path:
    with open(label_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if line and not line.lower().startswith("index"):
                bird_labels.append(line.split(",")[-1].strip() if "," in line else line)
print(f"Loaded {len(bird_labels)} labels from {label_path}")

In [ ]:
#@title Bird classifier inference helper { display-mode: "form" }

def load_mono(path: str, target_sr: int) -> np.ndarray:
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    return y.astype(np.float32)


def pad_crop(w: np.ndarray, n: int) -> np.ndarray:
    if len(w) >= n:
        return w[:n]
    out = np.zeros(n, dtype=np.float32)
    out[: len(w)] = w
    return out


def bird_topk(waveform: np.ndarray, k: int = 15):
    w = pad_crop(waveform, BIRD_SAMPLES)
    feed = {BIRD_WAVE_KEY: tf.constant(w[None, :], dtype=tf.float32)}
    for key, tensor in spec_in.items():
        if key == BIRD_WAVE_KEY:
            continue
        sh = tensor.shape.as_list()
        static = [1 if d is None else int(d) for d in sh]
        feed[key] = tf.zeros(static, dtype=tensor.dtype)
    out = bird_infer(**feed)
    # logits
    logits = None
    for name in ("logits", "predictions", "output_0"):
        if name in out:
            logits = out[name].numpy().reshape(-1)
            break
    if logits is None:
        for t in out.values():
            arr = t.numpy().reshape(-1)
            if 10 < arr.size < 100_000:
                logits = arr
                break
    p = np.exp(logits - logits.max())
    p = p / p.sum()
    order = np.argsort(-p)[:k]
    rows = []
    for rank, j in enumerate(order, 1):
        label = bird_labels[j] if j < len(bird_labels) else f"class_{j}"
        rows.append({"rank": rank, "label": label, "confidence": float(p[j])})
    return rows


MUSTELID_TOKENS = re.compile(
    r"mink|mustel|neogale|neovison|vison|weasel|marten|otter", re.I
)


def scan_mustelid(rows):
    return [r for r in rows if MUSTELID_TOKENS.search(r["label"] or "")]

## 3. Model B — Perch 2.0 (`perch_v2`)

Multi-taxa logits + embeddings. See [Perch-Hoplite](https://github.com/google-research/perch-hoplite).

In [ ]:
#@title Load Perch 2.0 { display-mode: "form" }
from perch_hoplite.zoo import model_configs

perch = model_configs.load_model_by_name("perch_v2")
PERCH_SR = 32000
PERCH_SECONDS = 5
print("Perch 2.0 loaded.")

In [ ]:
#@title Perch inference helper { display-mode: "form" }

def perch_topk(waveform: np.ndarray, k: int = 20):
    target = PERCH_SR * PERCH_SECONDS
    w = pad_crop(waveform, target)
    outputs = perch.embed(w)
    logits = outputs.logits.get("label") if hasattr(outputs, "logits") else None
    if logits is None:
        return [], getattr(outputs, "embeddings", None)
    logits = np.asarray(logits).reshape(-1)
    p = np.exp(logits - logits.max())
    p = p / (p.sum() + 1e-12)
    order = np.argsort(-p)[:k]
    # Label names: model may expose class_list — fall back to index
    names = getattr(perch, "class_list", None) or getattr(perch, "labels", None)
    rows = []
    for rank, j in enumerate(order, 1):
        label = names[j] if names is not None and j < len(names) else f"class_{j}"
        rows.append({"rank": rank, "label": str(label), "confidence": float(p[j])})
    emb = np.asarray(outputs.embeddings).reshape(-1) if hasattr(outputs, "embeddings") else None
    return rows, emb

## 4. Audio clips

Upload **mink vocalization** WAV/MP3 files. Add riparian **controls** without mink.

Document sources (Macaulay, field recordings, literature supplements) in the table below.

In [ ]:
#@title Upload audio { display-mode: "form" }
from google.colab import files
import pathlib

UPLOAD_DIR = pathlib.Path("/content/mink_study_audio")
UPLOAD_DIR.mkdir(exist_ok=True)
print("Select WAV/MP3/FLAC files (mink positives + controls):")
uploaded = files.upload()
paths = []
for name, data in uploaded.items():
    p = UPLOAD_DIR / name
    p.write_bytes(data)
    paths.append(str(p))
print("Saved:", paths)
if not paths:
    print("No files uploaded — using 5s silence sanity check only.")
    paths = []

In [ ]:
#@title Run OOTB comparison on all clips { display-mode: "form" }
import pandas as pd

records = []
clips = paths or ["__silence__"]

for clip in clips:
    if clip == "__silence__":
        bird_w = np.zeros(BIRD_SAMPLES, dtype=np.float32)
        perch_w = np.zeros(PERCH_SR * PERCH_SECONDS, dtype=np.float32)
        name = "silence_sanity"
    else:
        name = pathlib.Path(clip).name
        bird_w = load_mono(clip, target_sr=16000)
        perch_w = load_mono(clip, target_sr=PERCH_SR)

    b_rows = bird_topk(bird_w)
    p_rows, emb = perch_topk(perch_w)
    b_hit = scan_mustelid(b_rows)
    p_hit = scan_mustelid(p_rows)

    records.append({
        "clip": name,
        "bird_top1": b_rows[0]["label"] if b_rows else "",
        "bird_top1_conf": b_rows[0]["confidence"] if b_rows else 0,
        "bird_mustelid_hits": len(b_hit),
        "perch_top1": p_rows[0]["label"] if p_rows else "",
        "perch_top1_conf": p_rows[0]["confidence"] if p_rows else 0,
        "perch_mustelid_hits": len(p_hit),
    })

    print(f"\n=== {name} ===")
    print("Bird top-5:", [(r["label"], round(r["confidence"], 4)) for r in b_rows[:5]])
    if b_hit:
        print("Bird MUSTELID hits:", b_hit)
    print("Perch top-5:", [(r["label"], round(r["confidence"], 4)) for r in p_rows[:5]])
    if p_hit:
        print("Perch MUSTELID hits:", p_hit)

df = pd.DataFrame(records)
display(df)

## 5. Verdict (fill after run)

| Criterion | Bird classifier OOTB | Perch 2.0 OOTB |
|-----------|----------------------|----------------|
| Mink in top-5 on ≥70% positives | ☐ Yes ☐ No | ☐ Yes ☐ No |
| Confidence ≥ 0.2 on flagged label | ☐ Yes ☐ No | ☐ Yes ☐ No |

**Recommendation for 216labs:** *(one paragraph — ship / agile model / collect data)*

Copy results into `paper_draft.md` §7.

In [ ]:
#@title Auto-suggest verdict (heuristic) { display-mode: "form" }
if len(df) == 0:
    print("No results yet.")
else:
    bird_ok = (df["bird_mustelid_hits"] > 0).any()
    perch_ok = (df["perch_mustelid_hits"] > 0).any()
    print("Bird classifier OOTB mink/mustelid in top-k:", "YES" if bird_ok else "NO")
    print("Perch 2.0 OOTB mink/mustelid in top-k:", "YES" if perch_ok else "NO")
    if not bird_ok and not perch_ok:
        print(
            "\nSuggested: OOTB detection NOT supported — use Perch embeddings + agile modeling "
            "or fine-tune a linear head on labeled mink clips."
        )